# 1. Introduction

This notebook builds a Logistic Regression model from scratch using only NumPy.
The project uses the Titanic dataset to predict passenger survival.

The notebook includes:

- Data loading
- Data exploration
- Data preprocessing
- Feature scaling
- Logistic Regression implementation using NumPy
- Model evaluation
- ROC Curve visualization
- Feature importance analysis

In [1]:
import pandas as pd
import numpy as np
import seaborn as sns
import matplotlib.pyplot as plt
from sklearn.compose import ColumnTransformer
from sklearn.model_selection import train_test_split
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import classification_report, confusion_matrix, accuracy_score, roc_auc_score, precision_score, recall_score, roc_curve, f1_score
from sklearn.preprocessing import StandardScaler

# 2. Load Dataset

This section loads the Titanic dataset using Pandas and checks:

- Statistical summaries
- Missing values
- Dataset structure

In [ ]:
df = pd.read_csv('../data/titanic.csv')

print(df.describe())
print(df.isnull().sum())

# 3. Data Exploration

This section explores relationships between features and survival.

The visualizations include:

- Survival rates by passenger category
- Age distribution
- Survival count across age groups
- Correlation heatmap

In [ ]:
numeric_features = ["age", "family_size", "fare"]
binary_features = ["sex", "1st_class", "2nd_class", "3rd_class"]

for features in df.columns:
    if features in binary_features:
        sns.barplot(x=features, y='survived', data=df)
        plt.title('Survival Rate by {features}'.format(features=features))
        plt.show()

# Age distribution
plt.figure(figsize=(8,5))
sns.histplot(df['age'], bins=30, kde=True)
plt.title('Age Distribution')
plt.show()

#Age group by survivors
viz_df = df.copy()
bins = np.arange(0, viz_df['age'].max() + 5, 5)
viz_df['age_group'] = pd.cut(viz_df['age'], bins=bins)
plt.figure(figsize=(12,6))

sns.countplot(
    data=viz_df,
    x='age_group',
    hue='survived'
)
plt.xticks(rotation=45)
plt.title('Survival Count by Age Group (5-Year Bins)')
plt.xlabel('Age Group')
plt.ylabel('Count')
plt.legend(title='Survived', labels=['No', 'Yes'])
plt.show()


# Correlation heatmap for numerical columns
numeric_df = df.select_dtypes(include=['int64', 'float64'])

plt.figure(figsize=(10,6))
sns.heatmap(numeric_df.corr(), annot=True, cmap='coolwarm')
plt.title('Correlation Heatmap')
plt.show()

# 4. Data Cleaning and Preprocessing

This section prepares the dataset for training.

The steps include:

- Selecting features
- Separating target variable
- Converting data to NumPy arrays


In [ ]:
# Define Features (X) and Target (Y)
X = df.drop(columns=['survived', '2nd_class', 'family_size'])
y = df['survived']

# Split the data (80% Training, 20% Testing)
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

numeric_features = ["age", "fare"]
binary_features = ["sex", "1st_class", "3rd_class"]

preprocessor = ColumnTransformer(
    transformers=[
        ("num", StandardScaler(), numeric_features),
    ],
    remainder="passthrough"
)

x_train_standardized = preprocessor.fit_transform(X_train)
x_test_standardized = preprocessor.transform(X_test)


# 5. Train Test Split Using NumPy

This section manually splits the dataset into:

- Training set (80%)
- Testing set (20%)